# 09 — Embedded Systems and IoT Security

## What This Is
IoT devices are the fastest-growing attack surface: routers, cameras, industrial PLCs, medical devices, and smart home systems. They run stripped-down Linux or RTOS with hardcoded credentials, unencrypted update mechanisms, and no security monitoring.

## Why It Matters
Mirai botnet (2016) compromised 600,000+ IoT devices using factory-default credentials, launching the largest DDoS ever recorded (1.2 Tbps). Triton/TRISIS (2017) targeted Schneider Electric safety controllers in a Saudi petrochemical plant — the first known attack designed to cause physical harm.

## Where the Community Stands
NIST IR 8259 and ENISA IoT Security Guidelines define baseline security requirements. Matter protocol provides standardised IoT security. ETSI EN 303 645 is the first mandatory IoT security standard (UK PSTI Act 2024). SBOM requirements are now regulatory in US and EU.

## Common IoT Vulnerabilities
- Hardcoded credentials (OWASP IoT #1)
- Unencrypted protocols (Telnet, HTTP, MQTT cleartext)
- No firmware signing
- Unrestricted network services
- Insufficient physical security

In [ ]:
import hashlib
import re
import json
import secrets
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# OWASP IoT Top 10 (2018)
OWASP_IOT_TOP10 = {
    'I1': ('Weak/Hardcoded Passwords',       'Factory-default or hardcoded credentials'),
    'I2': ('Insecure Network Services',       'Unnecessary or unprotected services exposed'),
    'I3': ('Insecure Ecosystem Interfaces',   'Web UI, API, mobile without proper auth/encryption'),
    'I4': ('Lack of Secure Update Mechanism','No firmware signing or rollback protection'),
    'I5': ('Using Outdated Components',       'Deprecated libraries/OS without CVE monitoring'),
    'I6': ('Insufficient Privacy Protection', 'Unnecessary personal data collection/retention'),
    'I7': ('Insecure Data Transfer/Storage',  'Cleartext protocols, unencrypted storage'),
    'I8': ('Lack of Device Management',       'No remote patching, monitoring, or incident response'),
    'I9': ('Insecure Default Settings',       'Defaults chosen for ease of use, not security'),
    'I10':('Lack of Physical Hardening',      'No protection against physical access attacks'),
}

for vuln_id, (name, desc) in OWASP_IOT_TOP10.items():
    print(f'  {vuln_id}: {name}')
    print(f'       {desc}')

In [ ]:
# IoT device security scanner (static analysis)
@dataclass
class IoTFinding:
    owasp_id: str
    severity: str
    finding:  str
    evidence: str

HARDCODED_CREDENTIAL_PATTERNS = [
    (r'password\s*=\s*["\'](?!\{)[^"\'>]{3,}["\']', 'Hardcoded password'),
    (r'admin:\$1\$',                                  'Hardcoded MD5 crypt hash'),
    (r'(?i)pass(?:word)?\s*=\s*["\']12345',           'Weak default password'),
    (r'telnet\s+\d+',                                 'Telnet service configured'),
    (r'http://(?!localhost)',                           'Unencrypted HTTP endpoint'),
    (r'ENABLE_SSH\s*=\s*0',                           'SSH disabled'),
]

def scan_firmware_config(config_text: str) -> List[IoTFinding]:
    findings = []
    for pattern, description in HARDCODED_CREDENTIAL_PATTERNS:
        matches = re.findall(pattern, config_text)
        for match in matches[:3]:  # cap at 3
            owasp = 'I1' if 'password' in description.lower() or 'hash' in description.lower() else 'I7'
            if 'telnet' in description.lower() or 'SSH' in description:
                owasp = 'I2'
            sev = 'Critical' if owasp == 'I1' else 'High'
            findings.append(IoTFinding(
                owasp_id=owasp, severity=sev,
                finding=description,
                evidence=match[:60]
            ))
    return findings

# Simulate a vulnerable IoT device config
VULNERABLE_CONFIG = """
system_name = MyRouter_v2
admin_password = 'admin123'
root_hash = admin:$1$xyz$abcdefghijklmnop
enable_telnet 23
update_url = http://update.vendor.com/fw
ENABLE_SSH = 0
mqtt_broker = mqtt://cloud.vendor.com:1883
api_key = 'hardcoded-api-key-abc123'
"""

findings = scan_firmware_config(VULNERABLE_CONFIG)
print(f'IoT Config Scan: {len(findings)} findings\n')
for f in findings:
    print(f'  [{f.severity}] {f.owasp_id} — {f.finding}')
    print(f'    Evidence: {f.evidence}')

## Secure Firmware Update
Firmware updates without signing are trivially exploitable (Mirai, Trickbot, and ASUS Live Update used this vector). Secure OTA requires: ECDSA signature verification, version downgrade prevention, encrypted channel, and staged rollout.

In [ ]:
# Secure OTA update pipeline simulation
import hmac

@dataclass
class FirmwarePackage:
    version:       str
    min_version:   str   # downgrade prevention
    sha256:        str
    signature:     bytes
    payload:       bytes

def build_firmware_package(payload: bytes, version: str, min_version: str,
                            signing_key: bytes) -> FirmwarePackage:
    sha256_hex = hashlib.sha256(payload).hexdigest()
    manifest   = json.dumps({'version': version, 'min_version': min_version,
                              'sha256': sha256_hex}).encode()
    signature  = hmac.new(signing_key, manifest, hashlib.sha256).digest()
    return FirmwarePackage(version=version, min_version=min_version,
                           sha256=sha256_hex, signature=signature, payload=payload)

def verify_and_install(pkg: FirmwarePackage, current_version: str,
                        verify_key: bytes) -> Dict:
    result = {'ok': False, 'reason': ''}

    # 1. Verify payload hash
    computed_sha256 = hashlib.sha256(pkg.payload).hexdigest()
    if computed_sha256 != pkg.sha256:
        result['reason'] = 'Hash mismatch — payload corrupted'
        return result

    # 2. Verify signature
    manifest  = json.dumps({'version': pkg.version, 'min_version': pkg.min_version,
                             'sha256': pkg.sha256}).encode()
    expected  = hmac.new(verify_key, manifest, hashlib.sha256).digest()
    if not hmac.compare_digest(expected, pkg.signature):
        result['reason'] = 'Signature invalid — reject update'
        return result

    # 3. Downgrade prevention
    def version_tuple(v): return tuple(int(x) for x in v.split('.'))
    if version_tuple(current_version) < version_tuple(pkg.min_version):
        result['reason'] = f'Downgrade blocked: current={current_version} min={pkg.min_version}'
        return result

    result['ok'] = True
    result['reason'] = f'Update accepted: {current_version} -> {pkg.version}'
    return result

SIGNING_KEY    = secrets.token_bytes(32)
CURRENT_VER    = '3.1.0'
FW_PAYLOAD     = bytes(range(256)) * 100  # simulated firmware blob

# Valid update
pkg = build_firmware_package(FW_PAYLOAD, '3.2.0', '3.0.0', SIGNING_KEY)
r1  = verify_and_install(pkg, CURRENT_VER, SIGNING_KEY)
print(f'Valid update:              {r1}')

# Tampered payload
tampered_pkg = FirmwarePackage(**{**pkg.__dict__, 'payload': b'EVIL' + FW_PAYLOAD[4:]})
r2 = verify_and_install(tampered_pkg, CURRENT_VER, SIGNING_KEY)
print(f'Tampered payload:          {r2}')

# Rollback attack
old_pkg = build_firmware_package(FW_PAYLOAD, '2.8.0', '2.5.0', SIGNING_KEY)
r3 = verify_and_install(old_pkg, CURRENT_VER, SIGNING_KEY)
print(f'Rollback to 2.8.0:         {r3}')

# Wrong signing key (compromised vendor)
evil_key = secrets.token_bytes(32)
evil_pkg = build_firmware_package(FW_PAYLOAD, '3.2.0', '3.0.0', evil_key)
r4 = verify_and_install(evil_pkg, CURRENT_VER, SIGNING_KEY)
print(f'Wrong signing key:         {r4}')

## IoT Security Baseline Scorecard
Maturity scoring against ETSI EN 303 645 requirements — the regulatory baseline for consumer IoT.

In [ ]:
ETSI_REQUIREMENTS = [
    ('P-1', 'No universal default passwords',               'Critical'),
    ('P-2', 'Implement vulnerability disclosure policy',    'High'),
    ('P-3', 'Keep software updated',                        'High'),
    ('P-4', 'Securely store sensitive parameters',          'Critical'),
    ('P-5', 'Communicate securely',                         'High'),
    ('P-6', 'Minimise attack surfaces',                     'Medium'),
    ('P-7', 'Ensure software integrity',                    'High'),
    ('P-8', 'Ensure personal data is protected',            'Medium'),
    ('P-9', 'Make systems resilient to outages',            'Medium'),
    ('P-10','Monitor system telemetry data',                'Low'),
    ('P-11','Make it easy to delete user data',             'Medium'),
    ('P-12','Make installation easy to secure',             'Medium'),
    ('P-13','Validate input data',                          'High'),
]

def score_iot_device(compliance: Dict[str, bool]) -> Dict:
    total   = len(ETSI_REQUIREMENTS)
    passed  = sum(1 for p, _, _ in ETSI_REQUIREMENTS if compliance.get(p, False))
    critical_fail = [p for p, _, sev in ETSI_REQUIREMENTS
                     if sev == 'Critical' and not compliance.get(p, False)]
    return {
        'passed':         passed,
        'total':          total,
        'score':          f'{passed}/{total} ({passed/total*100:.0f}%)',
        'critical_fails': critical_fail,
        'compliant':      passed == total,
    }

# Example device assessment
compliance = {
    'P-1': False,  # still has default admin/admin
    'P-2': True,   # has a disclosure policy
    'P-3': False,  # no OTA update
    'P-4': False,  # credentials in plaintext config
    'P-5': True,   # uses TLS
    'P-6': True,   # minimal services
    'P-7': False,  # no firmware signing
    'P-8': True,   # no personal data collected
    'P-9': True,   # failsafe boot
    'P-10':True,   # telemetry enabled
    'P-11':True,   # factory reset available
    'P-12':True,   # setup wizard
    'P-13':True,   # input validation
}

score = score_iot_device(compliance)
print('ETSI EN 303 645 Assessment:')
print(f'  Score: {score["score"]}')
print(f'  Compliant: {score["compliant"]}')
if score['critical_fails']:
    print(f'  CRITICAL failures (blocking): {score["critical_fails"]}')
print('\nRequirement details:')
for p, name, sev in ETSI_REQUIREMENTS:
    status = 'PASS' if compliance.get(p) else 'FAIL'
    print(f'  [{status}] {p} [{sev:<8}] {name}')